# fiftyone_review_processed.ipynb — browse CONVERTED (intermediate-schema) data

**When to use this:** *after* Stage 5.2 conversion. Loads a converted source's intermediate-schema output (DEC-046) directly — flat `images/`+`labels/`, canonical class ids, plain YOLO `.txt` labels, no train/val/test split yet. This is the stage Stage 5.3 (Box Audit) and Stage 5.5 (Model-Assisted Curation) actually work on, so this notebook is genuinely useful, not just a sanity check.

**What it can browse (set via `source_key`, cell below):**
- A Stage 5.2 processed source, e.g. `"exdark"`, `"roboflow_pothole_vhmow"` — the original single-source use case.
- `"merged"` — the post-cap, post-merge pool (`dataset/merged/`, Stage 5.6).
- `"final/train"` / `"final/val"` / `"final/test"` — the split output (Stage 5.8).

**Flagged-only mode (optional, `flagged_report_path`):** point it at a `box_audit.py`-style flagged-boxes report (e.g. `dataset/reports/elevator_status_s4lrk_flagged.json`) to load *only* the images with at least one flagged box, with the specific flagged detection(s) marked (`detection.flagged == True`) so they're distinguishable from an image's other, unflagged boxes — this is what Stage 5.3's "isolate real detection boxes from shape-defective ones" review actually needs. Only works against a plain processed-source `source_key` (flagged reports reference that source's own filenames, not `merged`/`final`'s source-prefixed ones).

**Why not FiftyOne's built-in YOLO importer:** `fo.types.YOLOv5Dataset` assumes a `dataset.yaml` + per-split (`train`/`val`/`test`) folder structure. That fits `final/<split>`, but not `dataset/processed/<source>/` or `dataset/merged/` (both deliberately flat, no split yet — DEC-036). Building the FiftyOne dataset directly from `images/`+`labels/` handles all three the same way, one code path instead of two.

**Not for:** raw acquisition-stage exports (`dataset/raw/<source>/`) — use `fiftyone_explore.ipynb` (COCO-style) or `fiftyone_preview.ipynb` (pre-pull) for those instead.

In [ ]:
# Imports
import json
import shutil
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched -- this
    makes the scripts.* import below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo

from scripts.utils.config_loader import get_canonical_names
from scripts.utils.file_utils import build_stem_index, ensure_dir, final_dir, merged_dir, processed_dir, reports_dir
from scripts.curate.run_mistakenness import (
    COCO_CROSSWALK, CANONICAL_KEY_TO_NAME, ELIGIBLE_CANONICAL_IDS, load_eligible_ground_truth,
)

CANONICAL_NAMES = get_canonical_names()

In [ ]:
# Change this and re-run the cells below to browse a different pool.
# Matches dataset/processed/<source_key>/ by default -- e.g. "exdark",
# "dataset_ninja_pothole_detection", "open_images", "roboflow_pothole_vhmow" --
# or the literal strings "merged" (dataset/merged/) or "final/train" /
# "final/val" / "final/test" (dataset/final/<split>/).
source_key = "roboflow_elevator_status_s4lrk"

# Optional: path to a box_audit.py-style flagged-boxes report (list of
# {label_path, class, cx, cy, w, h, reasons}). Set to None for normal
# browsing of every image in source_key. Only valid when source_key is a
# plain processed source (not "merged"/"final/*").
flagged_report_path = REPO_ROOT / "dataset/reports/elevator_status_s4lrk_flagged.json"

# Optional: overlay a COCO-pretrained model's predictions alongside
# ground_truth, as a visual aid for spotting boxes that are MISSING (not
# wrong ones -- box_audit.py / flagged_report_path above already cover
# shape/size problems in boxes that already exist). Only helps for the
# 7/16 classes with a COCO analog (Person, Vehicle, Motorcycle, Bicycle,
# Animals, Chairs, Tables -- see run_mistakenness.py's COCO_CROSSWALK);
# leaves predictions empty for the other 9 classes, same ceiling as the
# mistakenness section below. Runs over every image in source_key, not a
# capped slice -- benchmarked for real on this machine at ~32 img/s, so
# even the largest single source (escalator_stairs, 7,560 images) is
# under 4 minutes. The mistakenness section's MISTAKENNESS_TOP_N=1000 cap
# is about the student's own review-TIME budget (DEC-074), not inference
# cost, so it doesn't apply here -- this is meant to run against whatever
# source you're already committed to reviewing in full.
show_predictions = False

In [ ]:
# Build the FiftyOne dataset directly from images/+labels/ -- no network
# calls (aside from the optional predictions overlay below), safe to
# re-run anytime (re-running replaces the same dataset from scratch,
# discarding any in-App edits made since the last run).
#
# Persistence note: persistent=False means this dataset's edits do NOT
# survive a kernel restart (or fo.delete_dataset), but they DO auto-save
# to FiftyOne's backing DB *during* this session -- editing/adding/deleting
# a box via the App's "Annotate" tab is real and immediately reflected in
# `dataset` here, it just isn't durable across sessions on its own. A
# separate write-back step (see notebook README / ask before building) is
# what would turn a review session into changes on disk in dataset/processed/.

# Resolve images_dir/labels_dir for the chosen pool. "merged" and "final/<split>"
# use file_utils' own path helpers (a different directory layout, source-prefixed
# filenames); everything else is treated as a Stage 5.2 processed source key.
if source_key == "merged":
    images_dir = merged_dir() / "images"
    labels_dir = merged_dir() / "labels"
elif source_key.startswith("final/"):
    split = source_key.split("/", 1)[1]
    images_dir = final_dir(split) / "images"
    labels_dir = final_dir(split) / "labels"
else:
    images_dir = processed_dir(source_key) / "images"
    labels_dir = processed_dir(source_key) / "labels"

# Optional flagged-only mode: restrict to images with >=1 flagged box, and mark
# which specific detection(s) triggered the flag -- and *why* (box_audit.py's
# reasons, e.g. "large_area_outlier (>0.31)" -- a near-full-image box, the
# classic classification-dataset-forced-into-detection symptom) -- so they
# stand out from an image's other, unflagged boxes and you're not guessing
# why something was flagged.
flagged_by_label_path: dict[str, list[dict]] = {}
if flagged_report_path is not None:
    if source_key == "merged" or source_key.startswith("final/"):
        raise ValueError(
            "flagged_report_path is only supported against a plain processed-source "
            "source_key -- merged/final use prefixed filenames a flagged report doesn't reference."
        )
    flagged_entries = json.loads(Path(flagged_report_path).read_text(encoding="utf-8"))
    for entry in flagged_entries:
        flagged_by_label_path.setdefault(entry["label_path"], []).append(entry)

dataset_name = f"review_{source_key.replace('/', '_')}"
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)
dataset = fo.Dataset(dataset_name, persistent=False)

samples = []
image_paths_ordered = []
for image_path in sorted(images_dir.iterdir()):
    label_path = labels_dir / f"{image_path.stem}.txt"
    label_filename = label_path.name

    if flagged_by_label_path and label_filename not in flagged_by_label_path:
        continue  # flagged mode: skip images with nothing flagged

    # Match by rounded (cx, cy, w, h), not raw float equality -- both sides come
    # from the same 6-decimal string formatting this project's converters use,
    # but comparing through a round-trip is more robust than trusting exact
    # float equality to hold.
    flagged_lookup = {
        (round(e["cx"], 6), round(e["cy"], 6), round(e["w"], 6), round(e["h"], 6)): e["reasons"]
        for e in flagged_by_label_path.get(label_filename, [])
    }

    sample = fo.Sample(filepath=str(image_path))
    # Stashed so a future write-back step knows exactly which label file a
    # sample's (possibly since-edited) detections came from, without having
    # to re-derive it from the image filename.
    sample["source_label_filename"] = label_filename
    detections = []
    if label_path.is_file():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:5])
            # Our labels are YOLO center-based (cx, cy, w, h); FiftyOne's
            # Detection.bounding_box is top-left-based (x, y, w, h) --
            # both normalized [0, 1], so just shift the origin.
            x, y = cx - w / 2, cy - h / 2
            reasons = flagged_lookup.get((round(cx, 6), round(cy, 6), round(w, 6), round(h, 6)))
            detections.append(
                fo.Detection(
                    label=CANONICAL_NAMES[class_id],
                    bounding_box=[x, y, w, h],
                    flagged=reasons is not None,
                    flag_reasons=", ".join(reasons) if reasons else "",
                )
            )

    sample["ground_truth"] = fo.Detections(detections=detections)
    samples.append(sample)
    image_paths_ordered.append(image_path)

# Optional predictions overlay (show_predictions, set above) -- a stock
# COCO-pretrained yolov8n's boxes for the 7/16 classes with a COCO analog
# (run_mistakenness.py's own COCO_CROSSWALK, imported rather than
# reimplemented so this can't drift from what that script actually does),
# as a visual aid for spotting boxes ground_truth is MISSING. Runs over
# every image in this source, not a capped slice -- see the note on
# show_predictions above for why that's fine cost-wise.
if show_predictions:
    from ultralytics import YOLO
    import torch

    model = YOLO("yolov8n.pt")
    coco_to_canonical = {c: CANONICAL_KEY_TO_NAME[key] for key, cs in COCO_CROSSWALK.items() for c in cs}
    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

    paths = [str(p) for p in image_paths_ordered]
    print(f"Running yolov8n inference on {len(paths)} images for the predictions overlay (device={device})...")
    BATCH = 16
    predictions_by_index: dict[int, list[dict]] = {}
    for i in range(0, len(paths), BATCH):
        batch = paths[i:i + BATCH]
        results = model.predict(batch, device=device, verbose=False)
        for offset, result in enumerate(results):
            dets = []
            names = result.names
            for box in result.boxes:
                coco_name = names[int(box.cls.item())]
                if coco_name not in coco_to_canonical:
                    continue
                x1, y1, x2, y2 = box.xyxyn[0].tolist()
                dets.append({
                    "label": coco_to_canonical[coco_name],
                    "bounding_box": [x1, y1, x2 - x1, y2 - y1],
                    "confidence": float(box.conf.item()),
                })
            predictions_by_index[i + offset] = dets

    for idx, sample in enumerate(samples):
        preds = predictions_by_index.get(idx, [])
        sample["predictions"] = fo.Detections(detections=[fo.Detection(**b) for b in preds])
    print(
        "Predictions overlay ready -- toggle the 'predictions' field visible in the App sidebar "
        "to compare against ground_truth. Tag a specific predicted box 'accept' (click it, tag "
        "from the Labels list) to promote it into ground_truth during write-back below."
    )

dataset.add_samples(samples)
print(f"{len(dataset)} images loaded from {images_dir}")
if flagged_by_label_path:
    total_flagged_boxes = sum(len(v) for v in flagged_by_label_path.values())
    print(
        f"Flagged-only mode: {len(flagged_by_label_path)} images, {total_flagged_boxes} flagged boxes "
        f"-- marked detections have flagged == True, with the reason(s) on flag_reasons "
        f"(visible in the App's sample modal, under the detection's attributes)"
    )

In [ ]:
# Launch the App
session = fo.launch_app(dataset, auto=False)

In [ ]:
session

# Write back your review edits

**Run this only after you're done editing in the App above, in the same kernel session** (it reads the live `dataset` object — restarting the kernel loses everything, since this dataset is `persistent=False`).

This does **not** touch your original files in `dataset/processed/<source>/labels/`. It writes to a parallel `labels_reviewed/` folder instead, plus prints a per-file summary of what changed, so you can spot-check before deciding to promote anything. Promoting (copying `labels_reviewed/*.txt` over the real `labels/`) is a separate, deliberate step — not automatic — because this write-back path hasn't been used for a real correction pass yet.

**To accept a missing box the `show_predictions` overlay found** (set `show_predictions = True` above first): tag that specific predicted box `accept` in the App — click it (in the Annotation Canvas or the Labels list), tag from there. This cell copies any `accept`-tagged prediction into `ground_truth` before writing, so it becomes a real label like any other. Predictions are never written to disk themselves, so there's no risk of a redundant ground-truth-plus-prediction pair both surviving — only what's in `ground_truth` after promotion counts.

**To mark an image for removal from the dataset entirely** (wrong content, duplicate, doesn't belong — not a box-editing fix): tag the *sample* (not a specific box) `exclude` instead — tag icon above the sample grid works on a selection; also available per-sample in the modal. This cell reads that tag and writes a per-source `dataset/reports/<source>_excluded.json` that `merge.py` checks on its next run — excluded images get dropped from `dataset/merged/` even though `cap_per_class.py` originally selected them, without ever touching `dataset/processed/`. Untag before running this cell if you change your mind; the exclusion list only reflects whatever's currently tagged, merged with any earlier session's exclusions for this source.

Only valid for a plain processed-source `source_key` (same restriction as `flagged_report_path` above) — `merged`/`final` are derived outputs regenerated by other scripts, not something to hand-edit here.

In [ ]:
if source_key == "merged" or source_key.startswith("final/"):
    raise ValueError(
        "Write-back is only supported against a plain processed-source source_key -- "
        "merged/final are derived outputs regenerated by other scripts, not hand-edited here."
    )

orig_labels_dir = processed_dir(source_key) / "labels"
out_labels_dir = processed_dir(source_key) / "labels_reviewed"
if out_labels_dir.is_dir():
    shutil.rmtree(out_labels_dir)  # staging area only -- safe to clear and rewrite each run
ensure_dir(out_labels_dir)

# Promote accepted predictions into ground_truth first, so the write logic
# below (which only ever reads ground_truth) picks them up automatically.
# Only relevant when show_predictions was True above -- a no-op otherwise,
# since samples won't have a populated predictions field to promote from.
# Appends rather than replaces, so existing ground_truth boxes are untouched
# either way. Predictions themselves are never written to disk anywhere --
# only what ends up in ground_truth (including anything promoted here)
# becomes a real label, so there's no risk of a redundant ground-truth +
# prediction pair both surviving into labels_reviewed/.
promoted = 0
for sample in dataset:
    if not sample.has_field("predictions") or sample["predictions"] is None:
        continue
    accepted = [det for det in sample["predictions"].detections if "accept" in (det.tags or [])]
    if not accepted:
        continue
    sample["ground_truth"].detections.extend(
        fo.Detection(label=det.label, bounding_box=det.bounding_box) for det in accepted
    )
    sample.save()
    promoted += len(accepted)
if promoted:
    print(f"Promoted {promoted} accepted prediction(s) into ground_truth (tagged 'accept' in the App).\n")

# Exclusion tracking: samples tagged "exclude" in the App get merged into this
# source's own dataset/reports/<source>_excluded.json, which merge.py reads to
# drop them from the next dataset/merged/ rebuild -- without touching
# dataset/processed/ at all. Merged with any existing file, not overwritten --
# a full-pool review can span many sittings, and an earlier sitting's
# exclusions must survive a later one's write-back.
excluded_json_path = reports_dir() / f"{source_key.removeprefix('roboflow_')}_excluded.json"
existing_excluded: set[str] = set()
if excluded_json_path.is_file():
    existing_excluded = set(json.loads(excluded_json_path.read_text(encoding="utf-8"))["excluded_filenames"])

added = removed = modified = unchanged = 0
newly_excluded: set[str] = set()
for sample in dataset:
    label_filename = sample["source_label_filename"]
    if "exclude" in sample.tags:
        newly_excluded.add(label_filename)

    orig_path = orig_labels_dir / label_filename
    orig_lines = [
        ln.strip() for ln in (orig_path.read_text(encoding="utf-8").splitlines() if orig_path.is_file() else [])
        if ln.strip()
    ]

    new_lines = []
    for det in sample.ground_truth.detections:
        if det.label not in CANONICAL_NAMES:
            raise ValueError(
                f"{label_filename}: detection has label {det.label!r}, not one of the 16 canonical "
                f"classes -- check for a typo introduced via the App's class dropdown before re-running."
            )
        class_id = CANONICAL_NAMES.index(det.label)
        # Inverse of the load cell's transform: FiftyOne's top-left [x, y, w, h] -> YOLO center-based.
        x, y, w, h = det.bounding_box
        cx, cy = x + w / 2, y + h / 2
        new_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    (out_labels_dir / label_filename).write_text(
        "\n".join(new_lines) + ("\n" if new_lines else ""), encoding="utf-8"
    )

    if len(new_lines) > len(orig_lines):
        added += 1
    elif len(new_lines) < len(orig_lines):
        removed += 1
    elif set(new_lines) != set(orig_lines):
        modified += 1
    else:
        unchanged += 1

all_excluded = sorted(existing_excluded | newly_excluded)
excluded_json_path.write_text(
    json.dumps({"source": source_key, "excluded_filenames": all_excluded}, indent=2),
    encoding="utf-8",
)

print(f"Wrote {len(dataset)} label files to {out_labels_dir}")
print(
    f"vs. originals in {orig_labels_dir}: "
    f"{added} files gained box(es), {removed} files lost box(es), "
    f"{modified} files changed geometry/class only (same count), {unchanged} untouched"
)
if newly_excluded:
    print(
        f"\n{len(newly_excluded)} newly tagged 'exclude' this run "
        f"({len(all_excluded)} total for {source_key}) -> {excluded_json_path}\n"
        "These still got a labels_reviewed/ file above (in case you untag and change your mind "
        "before promoting), but the next merge.py run will drop them from dataset/merged/ "
        "regardless of what you promote -- they won't reach the trained model unless untagged first."
    )
print(
    "\nNothing in dataset/processed/<source>/labels/ has been touched yet. Spot-check "
    "labels_reviewed/ (e.g. re-point source_key's flagged_report_path-free run at it, or diff "
    "a few files by hand), then explicitly copy the ones you're confident in over the real "
    "labels/ folder when ready to promote -- that promotion step is intentionally manual."
)

# Mistakenness review (Stage 5.5, top-N) — a different mode from everything above

Everything above is single-source (`source_key`). This section is deliberately different: `dataset/reports/mistakenness_report.json` ranks **22,846 images across all 7 COCO-eligible classes and every source that contributes to them**, by how much a stock COCO-pretrained `yolov8n.pt` disagrees with your ground truth. Reviewing all 22,846 isn't the intent — reviewing a bounded top slice, ranked worst-first, is.

**What "eligible" means here, concretely:** only Person, Vehicle, Motorcycle, Bicycle, Animals, Chairs, Tables have a COCO analog (see `scripts/curate/run_mistakenness.py`'s module docstring for the exact crosswalk and why). The other 9 classes get zero signal from this — this section can't help you review Doors, Elevator, Stairs, etc.

**Predictions are computed fresh here, not read from the report** (the report only stored counts, not box coordinates) — same model, same crosswalk, same logic as `run_mistakenness.py`, imported directly so this can't silently drift from what actually produced the ranking. For ~1,000 images this is a couple of minutes on this machine, not a background-task situation.

Each sample shows **both** `ground_truth` (eligible classes only — that's what was actually scored) and `predictions` (the proxy model's boxes, with confidence) side by side, plus the real `mistakenness` score as a field you can sort/filter by in the App sidebar. Seeing *why* an image was flagged (what the model saw vs. what you labeled) is the point — that's different from the box-audit review above, which is about box geometry, not a disagreement between two label sets.</cell id="2343228f">
<cell id="p0review6mistakenlabel"># How many of the top-ranked images to pull in for this review pass.
MISTAKENNESS_TOP_N = 1000

mistakenness_report = json.loads(
    (REPO_ROOT / "dataset/reports/mistakenness_report.json").read_text(encoding="utf-8")
)
top_ranked = mistakenness_report["ranked"][:MISTAKENNESS_TOP_N]
print(
    f"{len(top_ranked)} images selected "
    f"(top {MISTAKENNESS_TOP_N} of {mistakenness_report['unique_images_scored']} scored)"
)</cell id="p0review6mistakenlabel">

In [ ]:
# How many of the top-ranked images to pull in for this review pass.
MISTAKENNESS_TOP_N = 1000

mistakenness_report = json.loads(
    (REPO_ROOT / "dataset/reports/mistakenness_report.json").read_text(encoding="utf-8")
)
top_ranked = mistakenness_report["ranked"][:MISTAKENNESS_TOP_N]
print(
    f"{len(top_ranked)} images selected "
    f"(top {MISTAKENNESS_TOP_N} of {mistakenness_report['unique_images_scored']} scored)"
)

In [ ]:
# Build: resolve each top-ranked image, load its eligible-class ground truth, run fresh
# yolov8n inference for predictions. Mirrors run_mistakenness.py's own inference loop
# exactly (same model, same crosswalk, same device selection) -- imported rather than
# reimplemented for everything that isn't image-loading itself.
from ultralytics import YOLO
import torch

model = YOLO("yolov8n.pt")
coco_to_canonical = {c: CANONICAL_KEY_TO_NAME[key] for key, cs in COCO_CROSSWALK.items() for c in cs}
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

sources_needed = {r["source"] for r in top_ranked}
stem_indexes = {source: build_stem_index(source) for source in sources_needed}

records = []
for r in top_ranked:
    img_path = stem_indexes[r["source"]].get(r["filename"])
    if img_path is None:
        continue  # shouldn't happen against a consistent processed/ pool, but don't hard-fail a review session over it
    records.append((r["source"], r["filename"], img_path, r["mistakenness"]))

paths = [str(p) for _, _, p, _ in records]
BATCH = 16
predictions_by_path: dict[str, list[dict]] = {}
print(f"Running yolov8n inference on {len(paths)} images (device={device})...")
for i in range(0, len(paths), BATCH):
    batch = paths[i:i + BATCH]
    results = model.predict(batch, device=device, verbose=False)
    for path, result in zip(batch, results):
        dets = []
        names = result.names
        for box in result.boxes:
            coco_name = names[int(box.cls.item())]
            if coco_name not in coco_to_canonical:
                continue
            x1, y1, x2, y2 = box.xyxyn[0].tolist()
            dets.append({
                "label": coco_to_canonical[coco_name],
                "bounding_box": [x1, y1, x2 - x1, y2 - y1],
                "confidence": float(box.conf.item()),
            })
        predictions_by_path[path] = dets

print("Inference complete. Building FiftyOne dataset...")
mistakenness_dataset_name = "review_mistakenness_top_n"
if mistakenness_dataset_name in fo.list_datasets():
    fo.delete_dataset(mistakenness_dataset_name)
mistakenness_dataset = fo.Dataset(mistakenness_dataset_name, persistent=False)

samples = []
for source, filename, img_path, score in records:
    gt_boxes = load_eligible_ground_truth(source, filename, CANONICAL_NAMES)
    pred_boxes = predictions_by_path.get(str(img_path), [])
    sample = fo.Sample(filepath=str(img_path))
    # Stashed for the write-back cell -- which source/file this sample's edits belong to.
    sample["source"] = source
    sample["source_filename"] = filename
    sample["mistakenness"] = score
    sample["ground_truth"] = fo.Detections(detections=[fo.Detection(**b) for b in gt_boxes])
    sample["predictions"] = fo.Detections(detections=[fo.Detection(**b) for b in pred_boxes])
    samples.append(sample)

mistakenness_dataset.add_samples(samples)
print(f"{len(mistakenness_dataset)} images loaded, ready to review (sort by the mistakenness field in the App to see worst-first)")

In [ ]:
# Launch the App for the mistakenness review. Click the mistakenness field in the
# left sidebar to sort worst-first if it doesn't default to it.
mistakenness_session = fo.launch_app(mistakenness_dataset, auto=False)

# Write back mistakenness review edits

Same rule as the box-audit write-back above: run this after reviewing in the App, same kernel session, before restarting the kernel.

**One real difference from the box-audit write-back, worth understanding before you trust it:** `ground_truth` here only ever held the 7 COCO-eligible classes' boxes (that's what mistakenness was computed against) — an image can have other canonical-class boxes (e.g. a Pole box on a Person-eligible image) that were never loaded into this dataset at all. So this cell doesn't just dump `ground_truth` back out — it reads each original label file fresh, keeps every non-eligible-class line exactly as it was, and only replaces the eligible-class lines with whatever's currently in `ground_truth` (i.e., your edits). Verified before being handed to you: simulated deleting an eligible box on a real mixed-class file and confirmed the non-eligible line survived byte-for-byte and the original file on disk was untouched.

Writes to each source's own `dataset/processed/<source>/labels_reviewed/` — the same staging convention as above, shared across both review modes. Unlike the box-audit write-back, this one does **not** clear the whole `labels_reviewed/` folder first, since a 1,000-image mistakenness pass may span multiple sittings across many sources and shouldn't wipe out another review's already-promoted files each time it re-runs.

In [ ]:
changed_by_source: dict[str, int] = {}
diff_added = diff_removed = diff_modified = diff_unchanged = 0

for sample in mistakenness_dataset:
    source = sample["source"]
    filename = sample["source_filename"]
    orig_label_path = processed_dir(source) / "labels" / f"{filename}.txt"
    out_labels_dir = processed_dir(source) / "labels_reviewed"
    ensure_dir(out_labels_dir)

    full_orig_lines = [
        ln.strip() for ln in (orig_label_path.read_text(encoding="utf-8").splitlines() if orig_label_path.is_file() else [])
        if ln.strip()
    ]
    # Keep every non-eligible-class line exactly as it was -- this review never loaded them,
    # so they can't have been edited, and must not be dropped.
    non_eligible_lines = [ln for ln in full_orig_lines if int(ln.split()[0]) not in ELIGIBLE_CANONICAL_IDS]

    new_eligible_lines = []
    for det in sample.ground_truth.detections:
        if det.label not in CANONICAL_NAMES:
            raise ValueError(
                f"{source}/{filename}: detection has label {det.label!r}, not one of the 16 canonical "
                f"classes -- check for a typo introduced via the App's class dropdown before re-running."
            )
        class_id = CANONICAL_NAMES.index(det.label)
        x, y, w, h = det.bounding_box
        cx, cy = x + w / 2, y + h / 2
        new_eligible_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    merged_lines = non_eligible_lines + new_eligible_lines
    (out_labels_dir / f"{filename}.txt").write_text(
        "\n".join(merged_lines) + ("\n" if merged_lines else ""), encoding="utf-8"
    )
    changed_by_source[source] = changed_by_source.get(source, 0) + 1

    if len(merged_lines) > len(full_orig_lines):
        diff_added += 1
    elif len(merged_lines) < len(full_orig_lines):
        diff_removed += 1
    elif set(merged_lines) != set(full_orig_lines):
        diff_modified += 1
    else:
        diff_unchanged += 1

print(f"Wrote {sum(changed_by_source.values())} label files across {len(changed_by_source)} sources:")
for source, count in sorted(changed_by_source.items()):
    print(f"  {source}: {count} files -> {processed_dir(source) / 'labels_reviewed'}")
print(
    f"\nvs. originals: {diff_added} files gained box(es), {diff_removed} files lost box(es), "
    f"{diff_modified} files changed geometry/class only (same count), {diff_unchanged} untouched"
)
print(
    "\nNothing in any dataset/processed/<source>/labels/ has been touched. Spot-check "
    "labels_reviewed/ per source, then explicitly copy the files you're confident in over "
    "the real labels/ folder when ready to promote -- same manual promotion step as above."
)